# Load versioned NCERT candidates into Supabase
Add the target project's Session pooler Postgres URL as a Kaggle secret named `SUPABASE_DB_URL`. This notebook creates a private `ncert_research` schema, validates passage and vector identity, writes one immutable candidate release in a transaction, and checks the stored count.


In [1]:
%pip -q install 'psycopg[binary]' pgvector

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json
from pathlib import Path
import numpy as np
import psycopg
from kaggle_secrets import UserSecretsClient
from pgvector.psycopg import register_vector

roots = Path('/kaggle/input')
contents = list(roots.rglob('ncert_prepared/candidate_passages.jsonl'))
vectors = list(roots.rglob('ncert_embeddings/candidate_vectors.npz'))
manifests = list(roots.rglob('ncert_embeddings/embedding_manifest.json'))
inventories = list(roots.rglob('ncert_inventory/manifest.json'))
gaps = list(roots.rglob('ncert_inventory/gaps.json'))
reports = list(roots.rglob('ncert_prepared/eda_report.json'))
assert len(contents) == len(vectors) == len(manifests) == len(inventories) == len(gaps) == len(reports) == 1, 'Attach one inventory, prepared release and matching embedding output'
inventory = json.loads(inventories[0].read_text())
report = json.loads(reports[0].read_text())
assert not json.loads(gaps[0].read_text()), 'Resolve inventory gaps before loading'
assert len(inventory) == report['source_chapters'] == 130
assert sum(row['pages'] for row in inventory) == report['physical_pages'] == 3066
assert len(report['by_grade_subject']) == 10
content_bytes = contents[0].read_bytes()
passages = [json.loads(line) for line in content_bytes.decode('utf-8').splitlines()]
assert {(p['source_file'], p['source_sha256']) for p in passages} == {(r['file'], r['sha256']) for r in inventory}
embedding = json.loads(manifests[0].read_text())
assert hashlib.sha256(content_bytes).hexdigest() == embedding['source_sha256']
assert hashlib.sha256(vectors[0].read_bytes()).hexdigest() == embedding['vectors_sha256']
with np.load(vectors[0]) as bundle:
    matrix = bundle['vectors']
    ids = bundle['passage_ids'].tolist()
assert matrix.shape == (len(passages), 1024) and ids == [p['passage_id'] for p in passages]
assert np.isfinite(matrix).all() and np.allclose(np.linalg.norm(matrix, axis=1), 1, atol=1e-5)
assert all(p['review_status'] == 'candidate' for p in passages)
source_fingerprint = hashlib.sha256(json.dumps(sorted({(p['source_file'], p['source_sha256']) for p in passages})).encode()).hexdigest()
release_id = hashlib.sha256((embedding['source_sha256'] + embedding['vectors_sha256'] + embedding['model_revision']).encode()).hexdigest()[:24]
url = UserSecretsClient().get_secret('SUPABASE_DB_URL')
assert url.startswith('postgresql://') or url.startswith('postgres://')


In [3]:
SCHEMA_SQL = "CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA extensions;\nCREATE SCHEMA IF NOT EXISTS ncert_research;\nREVOKE ALL ON SCHEMA ncert_research FROM PUBLIC, anon, authenticated;\n\nCREATE TABLE IF NOT EXISTS ncert_research.releases (\n    release_id text PRIMARY KEY,\n    source_manifest_sha256 text NOT NULL,\n    content_sha256 text NOT NULL,\n    embedding_model text NOT NULL,\n    embedding_revision text NOT NULL,\n    passage_count integer NOT NULL CHECK (passage_count >= 0),\n    review_status text NOT NULL CHECK (review_status IN ('candidate', 'approved')),\n    created_at timestamptz NOT NULL DEFAULT now()\n);\n\nCREATE TABLE IF NOT EXISTS ncert_research.passages (\n    release_id text NOT NULL REFERENCES ncert_research.releases(release_id),\n    passage_id text NOT NULL,\n    grade smallint NOT NULL CHECK (grade BETWEEN 6 AND 10),\n    subject text NOT NULL CHECK (subject IN ('maths', 'science')),\n    book_code text NOT NULL,\n    chapter smallint NOT NULL CHECK (chapter > 0),\n    physical_page integer NOT NULL CHECK (physical_page > 0),\n    source_file text NOT NULL,\n    source_sha256 text NOT NULL,\n    source_url text NOT NULL,\n    heading text NOT NULL,\n    content text NOT NULL CHECK (length(trim(content)) > 0),\n    review_flags text[] NOT NULL DEFAULT '{}',\n    review_status text NOT NULL CHECK (review_status IN ('candidate', 'approved')),\n    embedding extensions.vector(1024) NOT NULL,\n    PRIMARY KEY (release_id, passage_id)\n);\n\nREVOKE ALL ON ALL TABLES IN SCHEMA ncert_research FROM PUBLIC, anon, authenticated;\n"


In [4]:
with psycopg.connect(url, sslmode='require') as db:
    db.execute('CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA extensions')
    vector_schema = db.execute("SELECT n.nspname FROM pg_extension e JOIN pg_namespace n ON n.oid=e.extnamespace WHERE e.extname='vector'").fetchone()[0]
    db.execute(SCHEMA_SQL.replace('extensions.vector', psycopg.sql.Identifier(vector_schema).as_string(db) + '.vector'))
    db.commit()
    register_vector(db)
    with db.transaction():
        db.execute('INSERT INTO ncert_research.releases (release_id, source_manifest_sha256, content_sha256, embedding_model, embedding_revision, passage_count, review_status) VALUES (%s,%s,%s,%s,%s,%s,%s) ON CONFLICT (release_id) DO NOTHING',
                   (release_id, source_fingerprint, embedding['source_sha256'], embedding['model_id'], embedding['model_revision'], len(passages), 'candidate'))
        stored_release = db.execute('SELECT content_sha256, embedding_revision, passage_count FROM ncert_research.releases WHERE release_id=%s', (release_id,)).fetchone()
        assert stored_release == (embedding['source_sha256'], embedding['model_revision'], len(passages))
        with db.cursor() as cursor:
            for passage, vector in zip(passages, matrix):
                cursor.execute('INSERT INTO ncert_research.passages (release_id, passage_id, grade, subject, book_code, chapter, physical_page, source_file, source_sha256, source_url, heading, content, review_flags, review_status, embedding) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) ON CONFLICT (release_id, passage_id) DO NOTHING',
                               (release_id, passage['passage_id'], passage['grade'], passage['subject'], passage['book_code'], passage['chapter'], passage['page'], passage['source_file'], passage['source_sha256'], passage['source_url'], passage['heading'], passage['text'], passage['review_flags'], passage['review_status'], vector))
        count = db.execute('SELECT count(*) FROM ncert_research.passages WHERE release_id=%s', (release_id,)).fetchone()[0]
        assert count == len(passages), (count, len(passages))
print('Stored candidate release', release_id, 'with', count, 'passages')

Stored candidate release 9aea0ce28990f86074b2d813 with 7221 passages
